In [ ]:
# Cell 1: Install and Import Libraries
# Note: Using the 'openai' library for Azure OpenAI access
import os
import json
from openai import AzureOpenAI
from dotenv import load_dotenv # Recommended for managing secrets

# Load environment variables from .env file (for local development)
load_dotenv()

# --- AZURE AUTHENTICATION ---
# These variables should be set in your .env file or Azure Notebook environment
AZURE_API_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
DEPLOYMENT_NAME = "gpt-4-turbo" # Replace with your actual LLM deployment name (e.g., Llama3-70B)

try:
    # Cell 2: Initialize Azure Client
    client = AzureOpenAI(
        api_key=AZURE_API_KEY,
        azure_endpoint=AZURE_ENDPOINT,
        api_version="2024-02-15" # Check latest version
    )
    print("✅ Azure OpenAI client initialized successfully.")
    print(f"Targeting deployment: {DEPLOYMENT_NAME}")
except Exception as e:
    print(f"❌ Error initializing Azure client. Check credentials and endpoint. Error: {e}")

# Cell 3: Helper Function for API Call
# This function is the core interface to the single LLM.
def get_llm_response(role_prompt, user_input, max_tokens=1024, temperature=0.0):
    """Sends a request to the Azure-hosted LLM with a System Role Prompt."""
    messages = [
        {"role": "system", "content": role_prompt},
        {"role": "user", "content": user_input}
    ]
    
    # Configure the response format for JSON (crucial for Verifier)
    response_format = {"type": "text"}
    if "JSON" in role_prompt or "JSON" in user_input:
         response_format = {"type": "json_object"}
    
    try:
        response = client.chat.completions.create(
            model=DEPLOYMENT_NAME,
            messages=messages,
            max_tokens=max_tokens,
            temperature=temperature,
            response_format=response_format
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"LLM API Error: {e}"

In [ ]:
# Cell 4: Define Critical JSON Schema and Error Categories
VERIFIER_JSON_SCHEMA = """
{
  "valid": <boolean: true if final answer is correct, false otherwise>,
  "error_category": <string: one of 'CALCULATION_ERROR', 'CONCEPTUAL_FLAW', 'LOGIC_OMISSION', 'NONE'>,
  "critique_summary": <string: a brief, actionable explanation of the error>
}
"""

# Cell 5: Define Role Prompts (Simplified PoC Version)
PROVER_ROLE = (
    "You are the PROVER, an expert mathematician. Your goal is to generate a detailed, "
    "step-by-step Chain-of-Thought solution to the problem. Start with the initial setup. "
    "If given a CRITIQUE, strictly follow it to correct your next attempt."
)

VERIFIER_ROLE = (
    "You are the VERIFIER, a hyper-critical logic machine. Your sole task is to analyze the "
    "provided solution and output a JSON object strictly adhering to the specified schema. "
    "Analyze the solution for correctness and logic. ONLY output the JSON object. "
    f"Error categories are: 'CALCULATION_ERROR', 'CONCEPTUAL_FLAW', 'LOGIC_OMISSION', 'NONE'. "
    f"The required JSON schema is: {VERIFIER_JSON_SCHEMA}"
)

# Cell 6: Example Problem (from few-shot planning)
PROBLEM = (
    "A rectangular garden has sides in the ratio 4:3. If the area of the garden is 300 m^2, "
    "what is the length of the fence needed to enclose it? Provide your final answer as an integer."
)

# Cell 7: Example of a Flawed Solution (Prover's first attempt for PoC)
# This simulates the Prover making a calculation error: 2*(20+15) = 70, but the Prover will output 90.
FLAWED_SOLUTION_S1 = """
Solution:
1. Let the length be 4x and the width be 3x.
2. Area: (4x)(3x) = 12x^2.
3. 12x^2 = 300, so x^2 = 25, and x = 5.
4. Sides are 4(5)=20m and 3(5)=15m.
5. Perimeter P = 2(20 + 15) = 2(45) = 90m.
Final Answer: 90
"""

In [ ]:
# Cell 8: PCAF ITERATION 1 (Verifier Critique)
print("--- 1. VERIFIER CRITIQUE (JSON Output Proof) ---")

# The Verifier prompt includes the role, the problem, and the solution to critique.
verifier_input = (
    f"Problem: {PROBLEM}\n\n"
    f"Solution to Critique:\n{FLAWED_SOLUTION_S1}"
)

verifier_raw_output = get_llm_response(VERIFIER_ROLE, verifier_input, temperature=0.0)

try:
    # --- CRITICAL POINCE: JSON PARSING ---
    verifier_json = json.loads(verifier_raw_output)
    print("✅ JSON Parsing Successful. Verifier Output:")
    print(json.dumps(verifier_json, indent=2))
    
    # Check if the critique is valid and get correction signal
    if verifier_json.get('valid') == False:
        error_cat = verifier_json.get('error_category', 'UNKNOWN_ERROR')
        critique = verifier_json.get('critique_summary', 'No summary provided.')
        
        print("\n❌ Solution is invalid. Preparing Planner's Correction...")
        
        # --- PLANNER LOGIC (Correction Generation) ---
        planner_correction_hint = (
            f"CRITIQUE: The Verifier identified a **{error_cat}** at the final step. "
            f"Specifically: **{critique}**. You must rigorously re-examine your final calculation."
        )

        # --- NEXT PROVER INPUT ---
        prover_input_s2 = (
            f"ORIGINAL PROBLEM: {PROBLEM}\n\n"
            f"PREVIOUS FAILED ATTEMPT:\n{FLAWED_SOLUTION_S1}\n\n"
            f"PLANNER'S CORRECTION HINT:\n{planner_correction_hint}\n\n"
            f"--- GENERATE CORRECTED SOLUTION (ATTEMPT S2) ---"
        )
        
    else:
        print("\n✅ Solution passed. Loop terminates.")
        prover_input_s2 = None
        
except json.JSONDecodeError as e:
    print(f"❌ JSON DECODE FAILURE: LLM did not output clean JSON. Error: {e}")
    prover_input_s2 = None # Fail the loop

# Cell 9: PCAF ITERATION 2 (Prover Correction)
if prover_input_s2:
    print("\n--- 2. PROVER CORRECTION (Targeted Generation Proof) ---")
    
    # Send the corrected prompt to the single LLM instance
    prover_corrected_output = get_llm_response(PROVER_ROLE, prover_input_s2, temperature=0.0)
    
    print("Prover's Corrected Solution (S2):")
    print(prover_corrected_output)
    
    # You would typically run the Verifier again on S2, but for PoC, one loop is sufficient.
    print("\n[PoC Complete] The system successfully ran one full corrective loop.")
    print("The final correctness of S2 would be checked against the MathArena gold standard.")